SwiGLU-7 — production retraining-scope analysis

Load-only reporting for 12 fixed native-8K, one-billion-token trajectories: three retraining scopes at 20%, 30%, 40%, and 50% eligible-MLP parameter removal. This notebook reads completed artifacts only; it does not load models, datasets, or benchmark harnesses.

setup and canonical artifact paths

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'src' / 'mlp_replacement').is_dir()
)
RESULTS = PROJECT_ROOT / 'data/results/workflows/model/swiglu-7'
PREPARED_PATH = RESULTS / 'prepare-001/result.json'
STRATEGIES = ['S7-0', 'S7-1', 'S7-2']
TARGETS = [0.2, 0.3, 0.4, 0.5]
RUN_PATHS = {
    (strategy, target): RESULTS / f'{strategy}-target-{target:.1f}-run-001/result.json'
    for strategy in STRATEGIES for target in TARGETS
}
missing = [path for path in [PREPARED_PATH, *RUN_PATHS.values()] if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing canonical SwiGLU-7 artifacts:\n' + '\n'.join(map(str, missing)))
prepared = json.loads(PREPARED_PATH.read_text(encoding='utf-8'))
runs = {key: json.loads(path.read_text(encoding='utf-8')) for key, path in RUN_PATHS.items()}
if prepared.get('workflow') != 'swiglu-7' or prepared.get('status') != 'completed':
    raise ValueError('The shared SwiGLU-7 preparation is incomplete')
for (strategy, target), run in runs.items():
    if (run.get('workflow'), run.get('status')) != ('swiglu-7', 'completed'):
        raise ValueError(f'Incomplete run: {strategy} at {target:.0%}')
    if run.get('identity') != {'strategy': strategy, 'target': target}:
        raise ValueError(f'Artifact identity differs: {strategy} at {target:.0%}')
pd.set_option('display.max_columns', None)

fixed design and starting allocations

In [ ]:
configuration = prepared['configuration']
design_rows = []
for strategy, values in configuration['strategies'].items():
    design_rows.append({
        'strategy': strategy,
        'scope': values['trainable_scope'],
        'lora_rank': values.get('lora', {}).get('rank'),
        'learning_rate': configuration['recovery']['learning_rate'],
        'sequence_length': configuration['recovery']['sequence_length'],
        'effective_batch_tokens': configuration['recovery']['effective_batch_tokens'],
        'target_tokens': configuration['recovery']['target_tokens'],
    })
start_rows = []
for target, record in prepared['results']['starts'].items():
    start_rows.append({
        'eligible_mlp_removal': float(target),
        'construction': record['construction'],
        'start_tokens': record['tokens_seen'],
        'realized_mlp_removal': record.get('allocation_solver', {}).get('realized_mlp_removal', float(target)),
        'replacement_layers': sum(not row.get('retains_dense_module', False) for row in record['allocation']),
    })
display(pd.DataFrame(design_rows))
display(pd.DataFrame(start_rows).sort_values('eligible_mlp_removal'))

recovery trajectories

In [ ]:
recovery_rows = []
for (strategy, target), run in runs.items():
    for row in run['results']['recovery']['validation_history']:
        recovery_rows.append({
            'strategy': strategy,
            'eligible_mlp_removal': target,
            'cumulative_tokens': row['actual_tokens'],
            'training_hours': row['training_seconds'] / 3600,
            'recovery_validation_kl': row['recovery_validation_kl'],
            'mean_train_kl': row['mean_train_kl'],
            'legacy_prefix_ppl': row.get('wikitext_validation', {}).get('perplexity'),
        })
recovery_df = pd.DataFrame(recovery_rows)
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
for (strategy, target), rows in recovery_df.groupby(['strategy', 'eligible_mlp_removal']):
    rows = rows.sort_values('cumulative_tokens')
    label = f'{strategy} / {target:.0%}'
    axes[0].plot(rows['cumulative_tokens'] / 1e9, rows['recovery_validation_kl'], label=label)
    axes[1].plot(rows['training_hours'], rows['recovery_validation_kl'], label=label)
axes[0].set(xlabel='Cumulative recovery tokens (B)', ylabel='Historical C4 validation KL')
axes[1].set(xlabel='Training time (hours)', ylabel='Historical C4 validation KL')
axes[0].legend(ncol=2, fontsize=8)
axes[1].legend(ncol=2, fontsize=8)
figure.tight_layout()
plt.show()

one-billion-token endpoints and differences from S7-0

In [ ]:
endpoint_df = (
    recovery_df.sort_values('cumulative_tokens')
    .groupby(['strategy', 'eligible_mlp_removal'], as_index=False)
    .tail(1)
)
controls = (
    endpoint_df[endpoint_df['strategy'] == 'S7-0'][['eligible_mlp_removal', 'recovery_validation_kl']]
    .rename(columns={'recovery_validation_kl': 's7_0_validation_kl'})
)
endpoint_df = endpoint_df.merge(controls, on='eligible_mlp_removal', validate='many_to_one')
endpoint_df['validation_kl_delta_vs_s7_0'] = (
    endpoint_df['recovery_validation_kl'] - endpoint_df['s7_0_validation_kl']
)
display(endpoint_df[[
    'strategy', 'eligible_mlp_removal', 'cumulative_tokens', 'training_hours',
    'recovery_validation_kl', 'validation_kl_delta_vs_s7_0', 'legacy_prefix_ppl',
]].sort_values(['eligible_mlp_removal', 'strategy']))

historical S7-0 comparison with SwiGLU-6 on their shared evaluation contexts

In [ ]:
control_rows = []
for target, historical in prepared['results']['swiglu_6_controls'].items():
    target = float(target)
    current = runs[('S7-0', target)]['results']['evaluation']
    prior = historical['metrics']
    for unit in ('validation-128', 'validation-2048', 'test-128', 'test-2048'):
        control_rows.append({
            'eligible_mlp_removal': target,
            'metric': f'{unit} perplexity',
            'swiglu_6': prior['likelihood'][unit]['perplexity'],
            's7_0': current['likelihood'][unit]['perplexity'],
        })
    control_rows.append({
        'eligible_mlp_removal': target,
        'metric': 'macro task accuracy',
        'swiglu_6': historical['comparison']['macro_accuracy'],
        's7_0': current['comparison_to_dense']['macro_accuracy'],
    })
control_df = pd.DataFrame(control_rows)
control_df['s7_0_minus_swiglu_6'] = control_df['s7_0'] - control_df['swiglu_6']
display(control_df)

full WikiText evaluation and deployment footprint

In [ ]:
likelihood_rows = []
footprint_rows = []
for (strategy, target), run in runs.items():
    evaluation = run['results']['evaluation']
    for unit, metrics in evaluation['likelihood'].items():
        dense_metrics = prepared['results']['dense_evaluation']['likelihood'][unit]
        likelihood_rows.append({
            'strategy': strategy, 'eligible_mlp_removal': target,
            'split': unit.split('-')[0], **metrics,
            'dense_perplexity': dense_metrics['perplexity'],
            'perplexity_delta_vs_dense': metrics['perplexity'] - dense_metrics['perplexity'],
        })
    footprint_rows.append({
        'strategy': strategy, 'eligible_mlp_removal': target,
        **evaluation['footprint'], **evaluation['resident_memory'],
        'bf16_conversion_ppl_delta': evaluation['bf16_conversion_ppl_delta'],
    })
likelihood_df = pd.DataFrame(likelihood_rows)
footprint_df = pd.DataFrame(footprint_rows)
display(likelihood_df[[
    'strategy', 'eligible_mlp_removal', 'split', 'context_length', 'stride',
    'predicted_tokens', 'loss', 'perplexity', 'dense_perplexity',
    'perplexity_delta_vs_dense',
]].sort_values(['eligible_mlp_removal', 'strategy', 'split', 'context_length']))
display(footprint_df[[
    'strategy', 'eligible_mlp_removal', 'parameters', 'whole_model_parameter_removal',
    'tensor_file_bytes', 'bundle_bytes', 'gpu_allocated_delta_bytes',
    'gpu_reserved_delta_bytes', 'host_rss_delta_bytes', 'bf16_conversion_ppl_delta',
]].sort_values(['eligible_mlp_removal', 'strategy']))

zero-shot tasks and paired differences against dense

In [ ]:
task_rows = []
macro_rows = []
for (strategy, target), run in runs.items():
    comparison = run['results']['evaluation']['comparison_to_dense']
    macro_rows.append({
        'strategy': strategy, 'eligible_mlp_removal': target,
        'macro_accuracy': comparison['macro_accuracy'],
        'macro_delta': comparison['macro_delta'],
    })
    for task, row in comparison['tasks'].items():
        task_rows.append({
            'strategy': strategy, 'eligible_mlp_removal': target, 'task': task,
            'accuracy': row['student'], 'dense_accuracy': row['dense'],
            'paired_delta': row['student_minus_dense'],
            'ci95_low': row['ci95'][0], 'ci95_high': row['ci95'][1],
            'examples': row['examples'],
        })
task_df = pd.DataFrame(task_rows)
macro_df = pd.DataFrame(macro_rows)
display(task_df.sort_values(['task', 'eligible_mlp_removal', 'strategy']))
display(macro_df.sort_values(['eligible_mlp_removal', 'strategy']))

quality and measured deployment footprint

In [ ]:
quality_df = (
    likelihood_df[(likelihood_df['split'] == 'test') & (likelihood_df['context_length'] == 8192)]
    .merge(footprint_df[[
        'strategy', 'eligible_mlp_removal', 'tensor_file_bytes', 'gpu_allocated_delta_bytes'
    ]], on=['strategy', 'eligible_mlp_removal'], validate='one_to_one')
    .merge(macro_df, on=['strategy', 'eligible_mlp_removal'], validate='one_to_one')
)
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
for row in quality_df.itertuples():
    label = f'{row.strategy}/{row.eligible_mlp_removal:.0%}'
    axes[0].scatter(row.tensor_file_bytes / 2**30, row.perplexity)
    axes[0].annotate(label, (row.tensor_file_bytes / 2**30, row.perplexity), fontsize=8)
    axes[1].scatter(row.gpu_allocated_delta_bytes / 2**30, 100 * row.macro_accuracy)
    axes[1].annotate(label, (row.gpu_allocated_delta_bytes / 2**30, 100 * row.macro_accuracy), fontsize=8)
axes[0].set(xlabel='BF16 tensor file size (GiB)', ylabel='Full WikiText test PPL (context 8192)')
axes[1].set(xlabel='Resident GPU allocated delta (GiB)', ylabel='Macro task accuracy (%)')
figure.tight_layout()
plt.show()

training cost and trainable-parameter scope

In [ ]:
cost_rows = []
group_rows = []
for (strategy, target), run in runs.items():
    recovery = run['results']['recovery']
    scope = run['results']['trainable_scope']
    cost_rows.append({
        'strategy': strategy, 'eligible_mlp_removal': target,
        'trainable_parameters': scope['trainable_parameters'],
        'trainable_fraction': scope['trainable_fraction'],
        'training_hours': recovery['training_seconds'] / 3600,
        'recovery_evaluation_hours': recovery['evaluation_seconds'] / 3600,
        'checkpoint_hours': recovery['checkpoint_seconds'] / 3600,
        'final_evaluation_hours': run['results']['final_evaluation_wall_seconds'] / 3600,
        'peak_gpu_allocated_gib': recovery['memory']['peak_vram_gib'],
    })
    for group in scope['groups']:
        group_rows.append({
            'strategy': strategy, 'eligible_mlp_removal': target,
            'parameter_group': group['name'], 'parameters': group['parameters'],
            'parameter_tensors': group['parameter_tensors'],
        })
cost_df = pd.DataFrame(cost_rows)
group_df = pd.DataFrame(group_rows)
display(cost_df.sort_values(['eligible_mlp_removal', 'strategy']))
display(group_df.sort_values(['eligible_mlp_removal', 'strategy', 'parameter_group']))

provenance and limits

Removal targets describe eligible MLP parameters; whole-model removal is reported separately. The grid isolates retraining scope under one fixed native-8K optimizer and token budget. SwiGLU-3 through SwiGLU-6 used 128-token recovery, so comparisons with SwiGLU-6 are historical rather than exact continuation controls; S7-0 is the internal control. Paired benchmark intervals describe evaluation-example uncertainty and do not establish training-seed uncertainty. Throughput and latency are not inferred from parameter count or FLOPs; only measured storage and resident memory are reported here.

In [ ]:
provenance_rows = [{
    'strategy': strategy,
    'eligible_mlp_removal': target,
    'run_fingerprint': run['run_fingerprint'],
    'prepared_sha256': run['prepared']['sha256'],
    'checkpoint_cleanup': run['results']['checkpoint_cleanup'],
    'status': run['status'],
} for (strategy, target), run in runs.items()]
display(pd.DataFrame(provenance_rows).sort_values(['eligible_mlp_removal', 'strategy']))
display({
    'preparation_fingerprint': prepared['run_fingerprint'],
    'source_hashes': prepared['provenance'],
    'evaluation_protocol_fingerprint': prepared['results']['evaluation_protocol']['protocol_fingerprint'],
})